In [ ]:
!nvidia-smi

#   Video Segmentation with MedSAM2
This notebook shows how to use MedSAM2 for video segmentation inference. 

If running locally using jupyter, first install `MedSAM2` in your environment using the [installation instructions](https://github.com/bowang-lab/MedSAM2?tab=readme-ov-file#installation) in the repository.


### '''To make the visual, make sure to convert the slices into JPEG for Images,GTs to run the inference'''
### '''To get a visua results, you have to convert the requireed slices into PNG otherwise it won't work as JPEG is raw'''

## Load Packages and MedsSAM2 video predictor

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sam2.build_sam import build_sam2_video_predictor

# helper functions
DAVIS_PALETTE = b"\x00\x00\x00\x80\x00\x00\x00\x80\x00\x80\x80\x00\x00\x00\x80\x80\x00\x80\x00\x80\x80\x80\x80\x80@\x00\x00\xc0\x00\x00@\x80\x00\xc0\x80\x00@\x00\x80\xc0\x00\x80@\x80\x80\xc0\x80\x80\x00@\x00\x80@\x00\x00\xc0\x00\x80\xc0\x00\x00@\x80\x80@\x80\x00\xc0\x80\x80\xc0\x80@@\x00\xc0@\x00@\xc0\x00\xc0\xc0\x00@@\x80\xc0@\x80@\xc0\x80\xc0\xc0\x80\x00\x00@\x80\x00@\x00\x80@\x80\x80@\x00\x00\xc0\x80\x00\xc0\x00\x80\xc0\x80\x80\xc0@\x00@\xc0\x00@@\x80@\xc0\x80@@\x00\xc0\xc0\x00\xc0@\x80\xc0\xc0\x80\xc0\x00@@\x80@@\x00\xc0@\x80\xc0@\x00@\xc0\x80@\xc0\x00\xc0\xc0\x80\xc0\xc0@@@\xc0@@@\xc0@\xc0\xc0@@@\xc0\xc0@\xc0@\xc0\xc0\xc0\xc0\xc0 \x00\x00\xa0\x00\x00 \x80\x00\xa0\x80\x00 \x00\x80\xa0\x00\x80 \x80\x80\xa0\x80\x80`\x00\x00\xe0\x00\x00`\x80\x00\xe0\x80\x00`\x00\x80\xe0\x00\x80`\x80\x80\xe0\x80\x80 @\x00\xa0@\x00 \xc0\x00\xa0\xc0\x00 @\x80\xa0@\x80 \xc0\x80\xa0\xc0\x80`@\x00\xe0@\x00`\xc0\x00\xe0\xc0\x00`@\x80\xe0@\x80`\xc0\x80\xe0\xc0\x80 \x00@\xa0\x00@ \x80@\xa0\x80@ \x00\xc0\xa0\x00\xc0 \x80\xc0\xa0\x80\xc0`\x00@\xe0\x00@`\x80@\xe0\x80@`\x00\xc0\xe0\x00\xc0`\x80\xc0\xe0\x80\xc0 @@\xa0@@ \xc0@\xa0\xc0@ @\xc0\xa0@\xc0 \xc0\xc0\xa0\xc0\xc0`@@\xe0@@`\xc0@\xe0\xc0@`@\xc0\xe0@\xc0`\xc0\xc0\xe0\xc0\xc0\x00 \x00\x80 \x00\x00\xa0\x00\x80\xa0\x00\x00 \x80\x80 \x80\x00\xa0\x80\x80\xa0\x80@ \x00\xc0 \x00@\xa0\x00\xc0\xa0\x00@ \x80\xc0 \x80@\xa0\x80\xc0\xa0\x80\x00`\x00\x80`\x00\x00\xe0\x00\x80\xe0\x00\x00`\x80\x80`\x80\x00\xe0\x80\x80\xe0\x80@`\x00\xc0`\x00@\xe0\x00\xc0\xe0\x00@`\x80\xc0`\x80@\xe0\x80\xc0\xe0\x80\x00 @\x80 @\x00\xa0@\x80\xa0@\x00 \xc0\x80 \xc0\x00\xa0\xc0\x80\xa0\xc0@ @\xc0 @@\xa0@\xc0\xa0@@ \xc0\xc0 \xc0@\xa0\xc0\xc0\xa0\xc0\x00`@\x80`@\x00\xe0@\x80\xe0@\x00`\xc0\x80`\xc0\x00\xe0\xc0\x80\xe0\xc0@`@\xc0`@@\xe0@\xc0\xe0@@`\xc0\xc0`\xc0@\xe0\xc0\xc0\xe0\xc0  \x00\xa0 \x00 \xa0\x00\xa0\xa0\x00  \x80\xa0 \x80 \xa0\x80\xa0\xa0\x80` \x00\xe0 \x00`\xa0\x00\xe0\xa0\x00` \x80\xe0 \x80`\xa0\x80\xe0\xa0\x80 `\x00\xa0`\x00 \xe0\x00\xa0\xe0\x00 `\x80\xa0`\x80 \xe0\x80\xa0\xe0\x80``\x00\xe0`\x00`\xe0\x00\xe0\xe0\x00``\x80\xe0`\x80`\xe0\x80\xe0\xe0\x80  @\xa0 @ \xa0@\xa0\xa0@  \xc0\xa0 \xc0 \xa0\xc0\xa0\xa0\xc0` @\xe0 @`\xa0@\xe0\xa0@` \xc0\xe0 \xc0`\xa0\xc0\xe0\xa0\xc0 `@\xa0`@ \xe0@\xa0\xe0@ `\xc0\xa0`\xc0 \xe0\xc0\xa0\xe0\xc0``@\xe0`@`\xe0@\xe0\xe0@``\xc0\xe0`\xc0`\xe0\xc0\xe0\xe0\xc0"
import numpy as np


CLASS_COLORS = {
    0: (0, 0, 0),       # Background
    1: (255, 0, 0),     # Femur
    2: (0, 255, 0),     # Tibia
    3: (0, 0, 255),     # Patella
    4: (255, 255, 0),   # Femoral Cartilage
    5: (255, 0, 255),   # Tibial Cartilage
    6: (0, 255, 255),   # Patellar Cartilage
    7: (180, 100, 100)  # Meniscus
}



def load_ann_png(path):
    """Load a PNG file as a mask and its palette."""
    mask = Image.open(path)
    palette = mask.getpalette()
    mask = np.array(mask).astype(np.uint8)
    return mask, palette

def get_per_obj_mask(mask):
    """Split a mask into per-object masks."""
    object_ids = np.unique(mask)
    object_ids = object_ids[object_ids > 0].tolist()
    per_obj_mask = {object_id: (mask == object_id) for object_id in object_ids}
    return per_obj_mask

def put_per_obj_mask(per_obj_mask, height, width):
    """Combine per-object masks into a single mask."""
    mask = np.zeros((height, width), dtype=np.uint8)
    object_ids = sorted(per_obj_mask)[::-1]
    for object_id in object_ids:
        object_mask = per_obj_mask[object_id]
        object_mask = object_mask.reshape(height, width)
        mask[object_mask] = object_id
    return mask

def load_masks_from_dir(input_mask_path):
    input_mask, input_palette = load_ann_png(input_mask_path)
    per_obj_input_mask = get_per_obj_mask(input_mask)

    return per_obj_input_mask, input_palette

def save_predictions_to_dir(
    output_mask_dir,
    video_name,
    frame_name,
    per_obj_output_mask,
    height,
    width,
):
    """Save masks to a directory as PNG files."""
    os.makedirs(os.path.join(output_mask_dir, video_name), exist_ok=True)

    output_mask = put_per_obj_mask(per_obj_output_mask, height, width)
    output_mask_path = os.path.join(
        output_mask_dir, video_name, f"{frame_name}.png"
    )
    assert output_mask.dtype == np.uint8
    assert output_mask.ndim == 2
    output_mask = Image.fromarray(output_mask)
    output_mask.save(output_mask_path)


def create_overlay(img_path, mask_path, palette):
    """Create an overlay of an image and a mask."""
    img = Image.open(img_path)
    mask = Image.open(mask_path)
    mask.putpalette(palette)
    mask_rgb = mask.convert("RGB")
    mask_rgb = mask_rgb.resize(img.size, Image.NEAREST)
    overlay = Image.blend(img, mask_rgb, alpha=0.5)
    return overlay



def mask_to_rgb(mask_img, class_colors):
    """Convert a label map (L or P mode) to an RGB color mask."""
    mask_np = np.array(mask_img)
    h, w = mask_np.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in class_colors.items():
        rgb[mask_np == cls] = color
    return Image.fromarray(rgb)


def create_overlay_colored(img_path, mask_path, class_colors, alpha=0.5):
    img = Image.open(img_path)
    img = ImageOps.exif_transpose(img).convert("RGB")

    mask = Image.open(mask_path)
    mask_rgb = mask_to_rgb(mask, class_colors)
    # mask.putpalette(palette)


    if mask_rgb.size != img.size:
        mask_rgb = mask_rgb.resize(img.size, Image.NEAREST)

    return Image.blend(img, mask_rgb, alpha=alpha)

    

## Passing middle slice as initial prompt

In [ ]:
# # change to customized path

import os


case ='9003175_00m_LEFT_SAG_3D_DESS_WE'

# # Just raw string paths
input_base_dir = f"/gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/V00_00m_test_1.0/{case}"
output_base_dir= f"/gpfs/home/machlm03/Segmentation/OAI_demo/Inference_test/{case}"
keyslice_root = f"{input_base_dir}/masks"

def get_middle_image(img_dir):
    """Return the filename of the middle image in the given directory."""
    valid_exts = {".jpg", ".jpeg", ".png"}
    imgs = sorted(
        [f for f in os.listdir(img_dir) if os.path.splitext(f)[1].lower() in valid_exts]
    )
    if not imgs:
        return None
    mid_idx = len(imgs) // 2
    return imgs[mid_idx]

# Example usage
keyslice = get_middle_image(keyslice_root)
print(f"Middle slice file: {keyslice}")



## Preparing a list of initial prompts

In [ ]:
import os
from pathlib import Path

def get_percentile_files(directory, percentiles=(30, 60), exts=(".png", ".jpg", ".jpeg")):
    """
    Given a directory, return the files at the given percentiles in sorted order.
    Works with a single int or a tuple/list of ints.
    """
    # Ensure percentiles is iterable
    if isinstance(percentiles, (int, float)):
        percentiles = [percentiles]

    directory = Path(directory)
    files = sorted([f for f in directory.iterdir() if f.suffix.lower() in exts])
    if not files:
        raise ValueError(f"No image files found in {directory}")

    selected_files = []
    for p in percentiles:
        idx = int(round((p / 100) * (len(files) - 1)))
        selected_files.append(files[idx])

    return selected_files


# Example usage:
mask_dir = f"/gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/V00_00m_test_1.0/{case}/masks"
initial_prompts = get_percentile_files(mask_dir)
print("Selected initial prompt files:")
for f in initial_prompts:
    print(f)


In [ ]:
VIDEO_DIR=input_base_dir
VIDEO_NAME="imgs_jpeg"


MODEL_CONFIG = "configs/sam2.1_hiera_t512.yaml"
MODEL_CHECKPOINT= "checkpoints/MedSAM2_latest.pt"
# MODEL_CHECKPOINT = ckpt_path


predictor = build_sam2_video_predictor(
    config_file=MODEL_CONFIG,
    ckpt_path=MODEL_CHECKPOINT,
    apply_postprocessing=True,
    # hydra_overrides_extra=hydra_overrides_extra,
    vos_optimized=  True,
)

# load the video frames
frame_names = [
        os.path.splitext(p)[0]
        for p in os.listdir(os.path.join(VIDEO_DIR, VIDEO_NAME))
        if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
    ]
frame_names = list(sorted(frame_names))
inference_state = predictor.init_state(
    video_path=os.path.join(VIDEO_DIR, VIDEO_NAME), async_loading_frames=False
)
height = inference_state["video_height"]
width = inference_state["video_width"]


## Prepare Inference and Add Initial Mask Prompt
### This part only takes single slice and match input ids

In [ ]:
output_base_dir

In [ ]:
keyslice = get_middle_image(keyslice_root)
print(f"Middle slice file: {keyslice}")

INITIAL_MASK_PROMPT = f"{keyslice_root}/{keyslice}"
OUTPUT_DIR=output_base_dir

input_palette = None

# Add input masks to MedSAM2 inference state before propagation
object_ids_set = None
input_frame_idx = int(keyslice.split(".")[0]) # using keyslice for input prompt
try:
    per_obj_input_mask, input_palette = load_masks_from_dir(input_mask_path=INITIAL_MASK_PROMPT)
except FileNotFoundError as e:
    raise RuntimeError(
        f"In {VIDEO_NAME=}, failed to load input mask for frame {input_frame_idx=}. "
        "Please add the `--track_object_appearing_later_in_video` flag "
        "for VOS datasets that don't have all objects to track appearing "
        "in the first frame (such as LVOS or YouTube-VOS)."
    ) from e

# get the list of object ids to track from the first input frame

# predictor.reset_state(inference_state)
bbox=[4,90,379,348]
predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=input_frame_idx,
    obj_id=object_id,
    box=bbox,
        )
    
# check and make sure we have at least one object to track
if object_ids_set is None or len(object_ids_set) == 0:
    raise RuntimeError(
        f"In {VIDEO_NAME=}, got no object ids on {input_frame_idx=}. "
        "Please add the `--track_object_appearing_later_in_video` flag "
        "for VOS datasets that don't have all objects to track appearing "
        "in the first frame (such as LVOS or YouTube-VOS)."
    )

In [ ]:
OUTPUT_DIR='/gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/test_bb'

In [ ]:
# run propagation throughout the video
OUTPUT_VIDEO_NAME="imgs"
os.makedirs(os.path.join(OUTPUT_DIR, OUTPUT_VIDEO_NAME), exist_ok=True)

video_segments = {}  # Store the per-frame segmentation results
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
    inference_state,
    reverse=True
):
    per_obj_output_mask = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, out_obj_id in enumerate(out_obj_ids)
    }
    video_segments[out_frame_idx] = per_obj_output_mask

# write the output masks
for out_frame_idx, per_obj_output_mask in video_segments.items():
    # save raw prediction results
    save_predictions_to_dir(
        output_mask_dir=OUTPUT_DIR,
        video_name=OUTPUT_VIDEO_NAME,
        frame_name=frame_names[out_frame_idx],
        per_obj_output_mask=per_obj_output_mask,
        height=height,
        width=width,
    )

In [ ]:
# # keyslice='132.png'
# INITIAL_MASK_PROMPT = f"{keyslice_root}/{keyslice}"
# INITIAL_MASK_PROMPT

In [ ]:
# load the video frames
frame_names = [
        os.path.splitext(p)[0]
        for p in os.listdir(os.path.join(VIDEO_DIR, VIDEO_NAME))
        if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
    ]
frame_names = list(sorted(frame_names))
inference_state = predictor.init_state(
    video_path=os.path.join(VIDEO_DIR, VIDEO_NAME), async_loading_frames=False
)
height = inference_state["video_height"]
width = inference_state["video_width"]
input_palette = None

# Add input masks to MedSAM2 inference state before propagation
object_ids_set = None
# input_frame_idx = 0  # use first frame as mask input
# try:
#     per_obj_input_mask, input_palette = load_masks_from_dir(input_mask_path=INITIAL_MASK_PROMPT)
# except FileNotFoundError as e:
#     raise RuntimeError(
#         f"In {VIDEO_NAME=}, failed to load input mask for frame {input_frame_idx=}. "
#         "Please add the `--track_object_appearing_later_in_video` flag "
#         "for VOS datasets that don't have all objects to track appearing "
#         "in the first frame (such as LVOS or YouTube-VOS)."
#     ) from e



In [ ]:
# per_obj_input_mask, input_palette = load_masks_from_dir("/gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/V00_00m_test/9298541_00m_RIGHT_SAG_3D_DESS_WE/masks/383.png")

## Percentile slices

In [ ]:

input_palette = None
object_ids_set = None
# Example: automatically pick 25%, 50%, 75% frames from mask dir
mask_dir = keyslice_root  # directory containing your GT masks
# mask_dir="/gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/V00_00m_test_1.0/9000099_00m_LEFT_SAG_3D_DESS_WE/masks"


# test_mask_dir="/gpfs/home/machlm03/Segmentation/OAI_demo/OAI_TrainTest/V00_00m_test_1.0/9003175_00m_LEFT_SAG_3D_DESS_WE/masks/"
initial_prompts_paths = get_percentile_files(mask_dir, percentiles=( 10,20 ,30,  50 , 60 ,70, 80,90,40))
initial_prompts_paths = get_percentile_files(mask_dir, percentiles=(70))


# Build (frame_idx, mask_path) pairs
initial_prompts = []

# Build (frame_idx, mask_path) pairs
initial_prompts = []
for mask_path in initial_prompts_paths:
    frame_idx = int(Path(mask_path).stem)  # assumes filename is frame number
    initial_prompts.append((frame_idx, mask_path))

# # Seed predictor with all prompts
object_ids_set = set()


for frame_idx, mask_path in initial_prompts:
    per_obj_input_mask, input_palette = load_masks_from_dir(mask_path)
    # print(per_obj_input_mask)
    for obj_id,obj_mask in per_obj_input_mask.items():
        # print(obj_id)
        # if obj_id in object_ids_set:
        #     continue
            
        object_ids_set.add(obj_id)
        # print(object_ids_set,frame_idx)
        predictor.add_new_mask(
            inference_state=inference_state,
            frame_idx=frame_idx,
            obj_id=obj_id,
            # mask=obj_mask,
            box=bbox,
        )
    print(object_ids_set,frame_idx)







        # bbox=[4,90,379,348]

        # predictor.add_new_points_or_box(
        #     inference_state=inference_state,
        #     frame_idx=input_frame_idx,
        #     obj_id=object_id,
        #     box=bbox,
        #         )




In [ ]:
# Show object IDs and count of True pixels for each
for obj_id, mask in per_obj_input_mask.items():
    true_count = mask.sum()  # number of True pixels
    print(f"Object ID: {obj_id}, True pixels: {true_count}")


In [ ]:
# get the list of object ids to track from the first input frame
if object_ids_set is None:
    object_ids_set = set(per_obj_input_mask)

In [ ]:
per_obj_input_mask.items()

In [ ]:

# for object_id, object_mask in per_obj_input_mask.items():
#     # check and make sure no new object ids appear only in later frames
#     if object_id not in object_ids_set:
#         raise RuntimeError(
#             f"In {VIDEO_NAME=}, got a new {object_id=} appearing only in a "
#             f"later {input_frame_idx=} (but not appearing in the first frame). "
#             "Please add the `--track_object_appearing_later_in_video` flag "
#             "for VOS datasets that don't have all objects to track appearing "
#             "in the first frame (such as LVOS or YouTube-VOS)."
#         )
#     predictor.add_new_mask(
#         inference_state=inference_state,
#         frame_idx=0,
#         obj_id=object_id,
#         mask=object_mask,
#             )




In [ ]:

    
# check and make sure we have at least one object to track
if object_ids_set is None or len(object_ids_set) == 0:
    raise RuntimeError(
        f"In {VIDEO_NAME=}, got no object ids on {input_frame_idx=}. "
        "Please add the `--track_object_appearing_later_in_video` flag "
        "for VOS datasets that don't have all objects to track appearing "
        "in the first frame (such as LVOS or YouTube-VOS)."
    )

## Run Inference

In [ ]:
OUTPUT_DIR

In [ ]:
# run propagation throughout the video
OUTPUT_VIDEO_NAME="imgs"
os.makedirs(os.path.join(OUTPUT_DIR, OUTPUT_VIDEO_NAME), exist_ok=True)

video_segments = {}  # Store the per-frame segmentation results
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
    inference_state,
    reverse=True
):
    per_obj_output_mask = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, out_obj_id in enumerate(out_obj_ids)
    }
    video_segments[out_frame_idx] = per_obj_output_mask

# write the output masks
for out_frame_idx, per_obj_output_mask in video_segments.items():
    # save raw prediction results
    save_predictions_to_dir(
        output_mask_dir=OUTPUT_DIR,
        video_name=OUTPUT_VIDEO_NAME,
        frame_name=frame_names[out_frame_idx],
        per_obj_output_mask=per_obj_output_mask,
        height=height,
        width=width,
    )

In [ ]:


# Reverse pass (backward from seed)
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
    inference_state,
    reverse=True
):
    per_obj_output_mask = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, out_obj_id in enumerate(out_obj_ids)
    }
    video_segments[out_frame_idx] = per_obj_output_mask

# Forward pass (from seed onward)
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
    inference_state,
    reverse=False
):
    per_obj_output_mask = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, out_obj_id in enumerate(out_obj_ids)
    }
    # Merge or overwrite depending on your logic
    if out_frame_idx in video_segments:
        video_segments[out_frame_idx].update(per_obj_output_mask)
    else:
        video_segments[out_frame_idx] = per_obj_output_mask

# write the output masks
for out_frame_idx, per_obj_output_mask in video_segments.items():
    # save raw prediction results
    save_predictions_to_dir(
        output_mask_dir=OUTPUT_DIR,
        video_name=OUTPUT_VIDEO_NAME,
        frame_name=frame_names[out_frame_idx],
        per_obj_output_mask=per_obj_output_mask,
        height=height,
        width=width,
    )


## Check for grey scales to match it later

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

mask_img = Image.open(INITIAL_MASK_PROMPT).convert("L")  # Convert to grayscale

# Convert to NumPy array
mask_arr = np.array(mask_img)

# Print unique values
unique_vals = np.unique(mask_arr)
print("Unique values:", unique_vals[:256], "... (total:", len(unique_vals), ")")

# Visualize
plt.imshow(mask_arr, cmap="gray")
plt.title("Grayscale Mask")
plt.axis("off")
plt.show()


## Visualize Inference Results
Visualize segmentation results for 3 key frames at the 25%, 50%, and 75% position in the sequence

In [ ]:
VIDEO_DIR

In [ ]:
import numpy as np
from PIL import Image, ImageOps
from pathlib import Path
import matplotlib.pyplot as plt

# Explicit mapping from grayscale to class-ID
GRAY_TO_CLASS = {
    0: 0,    # background
    60: 1,   # class 1
    120: 2,  # class 2
    180: 3,  # class 3
    240: 4,  # class 4
    44: 5,   # class 5
    164: 6,  # class 6
    104: 7,  # class 7
}

# Define class colors for visualization
CLASS_COLORS = {
    0: (0, 0, 0),       # background - black
    1: (255, 0, 0),     # class 1 - red
    2: (0, 255, 0),     # class 2 - green
    3: (0, 0, 255),     # class 3 - blue
    4: (255, 255, 0),   # class 4 - yellow
    5: (255, 0, 255),   # class 5 - magenta
    6: (0, 255, 255),   # class 6 - cyan
    7: (128, 128, 128), # class 7 - gray
}

def remap_with_lookup(mask_img):
    arr = np.array(mask_img, dtype=np.int16)
    lut = np.zeros(256, dtype=np.uint8)
    for gray, cls in GRAY_TO_CLASS.items():
        lut[gray] = cls
    return lut[arr]

def mask_ids_to_rgb(mask_ids, class_colors):
    h, w = mask_ids.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in class_colors.items():
        rgb[mask_ids == cls] = color
    return Image.fromarray(rgb)

def dice_iou(gt_bool, pred_bool):
    """Calculate Dice and IoU scores for boolean masks"""
    intersection = np.logical_and(gt_bool, pred_bool).sum()
    union = np.logical_or(gt_bool, pred_bool).sum()
    gt_sum = gt_bool.sum()
    pred_sum = pred_bool.sum()
    
    # Handle edge cases
    if gt_sum == 0 and pred_sum == 0:
        return 1.0, 1.0  # Perfect match when both are empty
    if union == 0:
        return 0.0, 0.0  # No union means no overlap
    
    dice = 2 * intersection / (gt_sum + pred_sum) if (gt_sum + pred_sum) > 0 else 0.0
    iou = intersection / union if union > 0 else 0.0
    
    return dice, iou

def list_images(directory):
    """List image files in directory"""
    extensions = ['.png', '.jpg', '.jpeg']
    files = []
    for ext in extensions:
        files.extend(directory.glob(f'*{ext}'))
        files.extend(directory.glob(f'*{ext.upper()}'))
    return [str(f) for f in files]

def overlay_image_mask(img_path, mask_path, class_colors, alpha=0.5):
    """Overlay a single mask (GT or prediction) on the original image."""
    img = Image.open(img_path).convert("RGB")
    img = ImageOps.exif_transpose(img)
    mask = Image.open(mask_path).convert("L")
    mask_rgb = mask_to_rgb(mask, class_colors)
    if mask_rgb.size != img.size:
        mask_rgb = mask_rgb.resize(img.size, Image.NEAREST)
    return Image.blend(img, mask_rgb, alpha=alpha)



mask_path = Path(f"{input_base_dir}/masks")       # GT masks
img_path = Path(f"{input_base_dir}/imgs_png")     # Original images
pre_masks_dir = Path(f"{OUTPUT_DIR}/imgs")        # Predicted masks

# OPTION 1: Evaluate all classes present in your data
# This will evaluate all non-background classes
labels = [1, 2, 3, 4, 5, 6, 7]  # All classes except background

# OPTION 2: Evaluate classes that actually appear in both GT and predictions
# Uncomment this section if you want to automatically detect common classes
"""
# Find common classes across all frames
all_gt_classes = set()
all_pred_classes = set()

frame_files = list_images(mask_path)
mask_files = list_images(pre_masks_dir)

frame_map = {Path(f).stem: f for f in frame_files}
mask_map = {Path(f).stem: f for f in mask_files}
common_frames = sorted(set(frame_map) & set(mask_map))

for k in common_frames:
    gt_file = mask_path / frame_map[k]
    pred_file = pre_masks_dir / mask_map[k]
    
    gt_mask_img = Image.open(gt_file).convert("L")
    pred_mask_img = Image.open(pred_file).convert("L")
    
    gt_ids = remap_with_lookup(gt_mask_img)
    pred_ids = remap_with_lookup(pred_mask_img)
    
    all_gt_classes.update(np.unique(gt_ids))
    all_pred_classes.update(np.unique(pred_ids))

# Use intersection of classes (classes present in both GT and predictions)
common_classes = all_gt_classes & all_pred_classes
labels = sorted([c for c in common_classes if c != 0])  # Exclude background
print(f"Common classes found: {labels}")
"""


mask_path = Path(f"{input_base_dir}/masks")       # GT masks
img_path = Path(f"{input_base_dir}/imgs_png")     # Original images
pre_masks_dir = Path(f"{OUTPUT_DIR}/imgs")        # Predicted masks



# --- Match frames ---
frame_files = list_images(mask_path)
mask_files = list_images(pre_masks_dir)

frame_map = {Path(f).stem: f for f in frame_files}
mask_map = {Path(f).stem: f for f in mask_files}
common = sorted(set(frame_map) & set(mask_map))
if not common:
    raise RuntimeError("No matching frame/mask filenames (by stem).")

# Select 3 key frames
num_frames = len(common)
sel_raw = [0.25, 0.55, 0.75]
selected_indices = [min(num_frames - 1, max(0, int(num_frames * r))) for r in sel_raw]
selected_keys = [common[i] for i in selected_indices]


# --- Main loop ---
dice_scores = []
iou_scores = []

# --- Main loop ---
dice_scores = []
iou_scores = []

plt.figure(figsize=(12, len(selected_keys) * 4))  # 2 columns, multiple rows

for row_idx, k in enumerate(selected_keys):
    img_file  = img_path / f"{k}.png"         # Original image
    gt_file   = mask_path / frame_map[k]      # GT mask
    pred_file = pre_masks_dir / mask_map[k]   # Predicted mask

    # Load GT and prediction masks
    gt_mask_img   = Image.open(gt_file).convert("L")
    pred_mask_img = Image.open(pred_file).convert("L")

    # Debug: show raw values before remapping
    gt_raw   = np.array(gt_mask_img)
    pred_raw = np.array(pred_mask_img)
    print(f"\nFrame: {k}")
    print(" GT raw values   :", sorted(np.unique(gt_raw)))
    print(" Pred raw values :", sorted(np.unique(pred_raw)))

    # Remap to class IDs
    gt_ids   = remap_with_lookup(gt_mask_img)
    pred_ids = remap_with_lookup(pred_mask_img)

    # Debug: show after remapping
    print(" GT remapped IDs :", sorted(np.unique(gt_ids)))
    print(" Pred remapped IDs:", sorted(np.unique(pred_ids)))

    # Show which classes from your labels are actually present
    gt_labels_present = [l for l in labels if l in gt_ids]
    pred_labels_present = [l for l in labels if l in pred_ids]
    print(f" GT has labels: {gt_labels_present}")
    print(f" Pred has labels: {pred_labels_present}")

    # Boolean masks for selected labels
    gt_bool   = np.isin(gt_ids, labels)
    pred_bool = np.isin(pred_ids, labels)
    
    # Show overlap info
    intersection_pixels = np.logical_and(gt_bool, pred_bool).sum()
    union_pixels = np.logical_or(gt_bool, pred_bool).sum()
    gt_pixels = gt_bool.sum()
    pred_pixels = pred_bool.sum()
    
    print(f" GT pixels: {gt_pixels}, Pred pixels: {pred_pixels}")
    print(f" Intersection: {intersection_pixels}, Union: {union_pixels}")

    # Compute metrics
    d, j = dice_iou(gt_bool, pred_bool)
    dice_scores.append(d)
    iou_scores.append(j)

    # Create overlays
    if img_file.exists():
        base_img = Image.open(img_file).convert("RGB")
        base_img = ImageOps.exif_transpose(base_img)
        
        # Resize masks to match image size if needed
        gt_rgb = mask_ids_to_rgb(gt_ids, CLASS_COLORS)
        pred_rgb = mask_ids_to_rgb(pred_ids, CLASS_COLORS)
        
        if gt_rgb.size != base_img.size:
            gt_rgb = gt_rgb.resize(base_img.size, Image.NEAREST)
        if pred_rgb.size != base_img.size:
            pred_rgb = pred_rgb.resize(base_img.size, Image.NEAREST)
        
        gt_overlay = Image.blend(base_img, gt_rgb, alpha=0.5)
        pred_overlay = Image.blend(base_img, pred_rgb, alpha=0.5)
    else:
        # If no base image, just show the masks
        gt_overlay = mask_ids_to_rgb(gt_ids, CLASS_COLORS)
        pred_overlay = mask_ids_to_rgb(pred_ids, CLASS_COLORS)

    # Plot GT overlay
    plt.subplot(len(selected_keys), 2, row_idx * 2 + 1)
    plt.imshow(gt_overlay)
    plt.title(f"{k} — GT\nClasses: {gt_labels_present}")
    plt.axis("off")

    # Plot Prediction overlay
    plt.subplot(len(selected_keys), 2, row_idx * 2 + 2)
    plt.imshow(pred_overlay)
    plt.title(f"Prediction\nClasses: {pred_labels_present}\nDice={d:.3f}, IoU={j:.3f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# Print mean scores
print(f"\n=== EVALUATION RESULTS ===")
print(f"Evaluated classes: {labels}")
print(f"Mean Dice over selected frames: {np.mean(dice_scores):.3f}")
print(f"Mean IoU over selected frames:  {np.mean(iou_scores):.3f}")

# Per-frame results
print(f"\nPer-frame results:")
for i, k in enumerate(selected_keys):
    print(f"Frame {k}: Dice={dice_scores[i]:.3f}, IoU={iou_scores[i]:.3f}")

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Explicit mapping from grayscale to class-ID
PALETTE_TO_CLASS = {
    0: 0,    # background
    60: 1,   # class 1
    120: 2,  # class 2
    180: 3,  # class 3
    240: 4,  # class 4
    44: 5,   # class 5
    164: 6,  # class 6
    104: 7,  # class 7
}


CLASS_NAMES = [
    "Background", "Femur", "Tibia", "Patella",
    "Femoral Cartilage", "Tibial Cartilage",
    "Patellar Cartilage", "Meniscus"
]


# --- Load mask and map palette values to class IDs ---
mask_raw = np.array(Image.open(INITIAL_MASK_PROMPT).convert("L"))
mask_ids = np.zeros_like(mask_raw, dtype=np.uint8)
for val, cls_id in PALETTE_TO_CLASS.items():
    mask_ids[mask_raw == val] = cls_id

# --- Map class IDs to RGB ---
rgb_mask = np.zeros((*mask_ids.shape, 3), dtype=np.uint8)
for cls_id, color in CLASS_COLORS.items():
    rgb_mask[mask_ids == cls_id] = color

# --- Create legend only for labels present and defined ---
unique_labels = np.unique(mask_ids)
patches = []
for l in unique_labels:
    l_int = int(l)  # convert np.uint8 to Python int
    if l_int in CLASS_COLORS and l_int < len(CLASS_NAMES):
        patches.append(
            mpatches.Patch(
                color=np.array(CLASS_COLORS[l_int]) / 255.0,
                label=CLASS_NAMES[l_int]
            )
        )

# --- Plot ---
plt.figure(figsize=(6, 6))
plt.imshow(rgb_mask)
plt.axis("off")
plt.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title("Mask with Class Colors")
plt.show()


In [ ]:
import numpy as np
from PIL import Image, ImageOps
from pathlib import Path
import matplotlib.pyplot as plt
from collections import defaultdict
import pandas as pd
from scipy.spatial.distance import directed_hausdorff





# Explicit mapping from grayscale to class-ID
GRAY_TO_CLASS = {
    0: 0,    # background
    60: 1,   # class 1
    120: 2,  # class 2
    180: 3,  # class 3
    240: 4,  # class 4
    44: 5,   # class 5
    164: 6,  # class 6
    104: 7,  # class 7
}

# Define class colors for visualization
CLASS_COLORS = {
    0: (0, 0, 0),       # background - black
    1: (255, 0, 0),     # class 1 - red
    2: (0, 255, 0),     # class 2 - green
    3: (0, 0, 255),     # class 3 - blue
    4: (255, 255, 0),   # class 4 - yellow
    5: (255, 0, 255),   # class 5 - magenta
    6: (0, 255, 255),   # class 6 - cyan
    7: (128, 128, 128), # class 7 - gray
}

# Class names for better reporting
CLASS_NAMES = {
    0: "Background",
    1: "Femur", 
    2: "Tibia",
    3: "Patella",
    4: "Femoral Cartilage",
    5: "Tibial Cartilage", 
    6: "Patellar Cartilage",
    7: "Meniscus"
}

def remap_with_lookup(mask_img):
    """Convert grayscale mask to class IDs using lookup table"""
    arr = np.array(mask_img, dtype=np.int16)
    lut = np.zeros(256, dtype=np.uint8)
    for gray, cls in GRAY_TO_CLASS.items():
        lut[gray] = cls
    return lut[arr]

def dice_iou(gt_bool, pred_bool):
    """Calculate Dice and IoU scores for boolean masks"""
    intersection = np.logical_and(gt_bool, pred_bool).sum()
    union = np.logical_or(gt_bool, pred_bool).sum()
    gt_sum = gt_bool.sum()
    pred_sum = pred_bool.sum()
    # Skip calculation if both masks are empty
    if gt_sum == 0 and pred_sum == 0:
        return None, None
    if union == 0:
        return 0.0, 0.0  # No union means no overlap
    dice = 2 * intersection / (gt_sum + pred_sum) if (gt_sum + pred_sum) > 0 else 0.0
    iou = intersection / union if union > 0 else 0.0
    return dice, iou


def list_images(directory):
    """List image files in directory"""
    extensions = ['.png', '.jpg', '.jpeg']
    files = []
    for ext in extensions:
        files.extend(directory.glob(f'*{ext}'))
        files.extend(directory.glob(f'*{ext.upper()}'))
    return [str(f) for f in files]


def hausdorff_distance_from_png(gt_mask, pred_mask):
    """
    Compute symmetric Hausdorff distance between two PNG masks.
    
    Parameters:
    - gt_path: Path to ground truth PNG mask
    - pred_path: Path to predicted PNG mask

    Returns:
    - float: Hausdorff distance
    """
    # Get foreground coordinates
    gt_coords = np.argwhere(gt_mask)
    pred_coords = np.argwhere(pred_mask)

    if gt_coords.size == 0 or pred_coords.size == 0:
        return float('inf')  # No foreground to compare

    # Directed distances
    d1 = directed_hausdorff(gt_coords, pred_coords)[0]
    d2 = directed_hausdorff(pred_coords, gt_coords)[0]

    return max(d1, d2)
    

In [ ]:

    
def compute_comprehensive_metrics(gt_dir, pred_dir, labels=None, verbose=True, save_results=None):
    print(gt_dir)
    """
    Compute comprehensive evaluation metrics for all files in directories.
    
    Parameters:
    -----------
    gt_dir : str or Path
        Directory containing ground truth masks
    pred_dir : str or Path  
        Directory containing predicted masks
    labels : list of int, optional
        List of class labels to evaluate. If None, evaluates all classes 1-7
    verbose : bool
        Whether to print detailed progress and results
    save_results : str or Path, optional
        Path to save detailed results CSV file
        
    Returns:
    --------
    dict : Comprehensive results dictionary containing:
        - overall_metrics: Mean Dice/IoU across all files and classes
        - per_class_metrics: Mean Dice/IoU per class
        - per_file_metrics: Dice/IoU for each file
        - class_statistics: Pixel counts and presence statistics per class
    """
    
    gt_dir = Path(gt_dir)
    pred_dir = Path(pred_dir)
    
    if labels is None:
        labels = [1, 2, 3, 4, 5, 6, 7]  # All non-background classes
    
    # Match files between GT and prediction directories
    gt_files = list_images(gt_dir)
    pred_files = list_images(pred_dir)
    # print(gt_files)
    # print(pred_dir)
    
    gt_map = {Path(f).stem: f for f in gt_files}
    pred_map = {Path(f).stem: f for f in pred_files}
    common_files = sorted(set(gt_map.keys()) & set(pred_map.keys()))
    
    if not common_files:
        raise RuntimeError("No matching files found between GT and prediction directories")
    
    if verbose:
        print(f"Found {len(common_files)} matching files")
        print(f"Evaluating classes: {labels}")
        print(f"Class names: {[CLASS_NAMES[l] for l in labels]}")
    
    # Initialize result storage
    all_dice_scores = []
    all_iou_scores = []
    per_class_dice = defaultdict(list)
    per_class_iou = defaultdict(list)
    per_class_hd=defaultdict(list)
    per_file_results = []
    class_pixel_stats = defaultdict(lambda: {'gt_pixels': 0, 'pred_pixels': 0, 'files_present_gt': 0, 'files_present_pred': 0})
    
    # Process each file
    for i, filename in enumerate(common_files):
        if verbose and (i + 1) % 50 == 0:
            print(f"Processing file {i+1}/{len(common_files)}: {filename}")
            
        gt_file = gt_dir / gt_map[filename]
        pred_file = pred_dir / pred_map[filename]
        
        # Load and convert masks
        try:
            gt_mask_img = Image.open(gt_file).convert("L")
            pred_mask_img = Image.open(pred_file).convert("L")
            
            gt_ids = remap_with_lookup(gt_mask_img)
            pred_ids = remap_with_lookup(pred_mask_img)
            
        except Exception as e:
            if verbose:
                print(f"Error processing {filename}: {e}")
            continue
        
        # File-level metrics storage
        file_dice_scores = []
        file_iou_scores = []
        all_hd_score=[]
        file_class_results = {}
        file_hd_scores=[]


        for class_label in labels:
            gt_bool = (gt_ids == class_label)
            pred_bool = (pred_ids == class_label)
        
            if not gt_bool.any() and not pred_bool.any():
                continue  # Skip this class if both masks are entirely background
        
            dice, iou = dice_iou(gt_bool, pred_bool)
            if dice is None or iou is None:
                continue

    # Continue your storage code

        
        # Process each class
        for class_label in labels:
            gt_bool = (gt_ids == class_label)
            pred_bool = (pred_ids == class_label)
            
            # Update pixel statistics
            class_pixel_stats[class_label]['gt_pixels'] += gt_bool.sum()
            class_pixel_stats[class_label]['pred_pixels'] += pred_bool.sum()
            if gt_bool.any():
                class_pixel_stats[class_label]['files_present_gt'] += 1
            if pred_bool.any():
                class_pixel_stats[class_label]['files_present_pred'] += 1
            
            # Calculate metrics
            dice, iou = dice_iou(gt_bool, pred_bool)
            hd_score=hausdorff_distance_from_png(gt_mask_img, pred_mask_img)
            
            # Store results
            per_class_dice[class_label].append(dice)
            per_class_iou[class_label].append(iou)
            per_class_hd[class_label].append(hd_score)
            file_dice_scores.append(dice)
            file_iou_scores.append(iou)
            file_hd_scores.append(hd_score)
            file_class_results[class_label] = {'dice': dice, 'iou': iou,'hd':hd_score}
        
        # Store file-level results
        file_mean_dice = np.mean(file_dice_scores)
        file_mean_iou = np.mean(file_iou_scores)
        file_mean_hd=np.mean(file_hd_scores)
        all_dice_scores.append(file_mean_dice)
        all_iou_scores.append(file_mean_iou)
        all_hd_score.append(file_hd_scores)
        
        per_file_results.append({
            'filename': filename,
            'mean_dice': file_mean_dice,
            'mean_iou': file_mean_iou,
            "mean_hd":file_mean_hd,
            'class_results': file_class_results
        })
    
    # Calculate final metrics
    overall_mean_dice = np.mean(all_dice_scores)
    overall_mean_iou = np.mean(all_iou_scores)
    overall_std_dice = np.std(all_dice_scores)
    overall_std_iou = np.std(all_iou_scores)
    
    
    # Per-class metrics
    per_class_results = {}
    for class_label in labels:
        if per_class_dice[class_label]:
            class_mean_dice = np.mean(per_class_dice[class_label])
            class_mean_iou = np.mean(per_class_iou[class_label])
            class_std_dice = np.std(per_class_dice[class_label])
            class_std_iou = np.std(per_class_iou[class_label])
        else:
            class_mean_dice = class_mean_iou = class_std_dice = class_std_iou = 0.0

    
            

        if per_class_hd[class_label]:
            class_mean_hd = np.mean(per_class_hd[class_label])
        else:
            class_mean_hd = 0.0
        
        per_class_results[class_label] = {
            'mean_dice': class_mean_dice,
            'mean_iou': class_mean_iou,
            'mean_hd': class_mean_hd,
            'std_dice': class_std_dice,
            'std_iou': class_std_iou,
            'class_name': CLASS_NAMES[class_label]
        }


    
    # Compile final results
    results = {
        'overall_metrics': {
            'mean_dice': overall_mean_dice,
            'mean_iou': overall_mean_iou,
            'std_dice': overall_std_dice,
            'std_iou': overall_std_iou,
            'num_files': len(common_files),
            'num_classes': len(labels)
        },
        'per_class_metrics': per_class_results,
        'per_file_metrics': per_file_results,
        'class_statistics': dict(class_pixel_stats)
    }
    
    # Print results
    if verbose:
        print_comprehensive_results(results, labels)
    
    # Save results to CSV if requested
    if save_results:
        save_results_to_csv(results, save_results, labels)
    
    return results

def print_comprehensive_results(results, labels):
    """Print comprehensive results in a formatted way"""
    print("\n" + "="*80)
    print("COMPREHENSIVE EVALUATION RESULTS")
    print("="*80)
    
    # Overall metrics
    overall = results['overall_metrics']
    print(f"\nOVERALL METRICS ({overall['num_files']} files, {overall['num_classes']} classes):")
    print(f"Mean Dice: {overall['mean_dice']:.4f} ± {overall['std_dice']:.4f}")
    print(f"Mean IoU:  {overall['mean_iou']:.4f} ± {overall['std_iou']:.4f}")
    print(f"Mean HD:  {overall['class_mean_hd']:.4f} ± {overall['std_iou']:.4f}")
    
    # Per-class metrics
    print(f"\nPER-CLASS METRICS:")
    print(f"{'Class':<20} {'Dice':<12} {'IoU':<12} {'GT Files':<10} {'Pred Files':<10}")
    print("-" * 70)
    
    for class_label in labels:
        class_result = results['per_class_metrics'][class_label]
        class_stats = results['class_statistics'][class_label]
        
        print(f"{class_result['class_name']:<20} "
              f"{class_result['mean_dice']:.4f}±{class_result['std_dice']:.3f}  "
              f"{class_result['mean_iou']:.4f}±{class_result['std_iou']:.3f}  "
              f"{class_stats['files_present_gt']:<10} "
              f"{class_stats['files_present_pred']:<10}")
    
    # # Class statistics
    # print(f"\nCLASS PIXEL STATISTICS:")
    # print(f"{'Class':<20} {'GT Pixels':<12} {'Pred Pixels':<12} {'Ratio':<10}")
    # print("-" * 60)
    
    # for class_label in labels:
    #     class_stats = results['class_statistics'][class_label]
    #     class_name = results['per_class_metrics'][class_label]['class_name']
    #     ratio = class_stats['pred_pixels'] / max(class_stats['gt_pixels'], 1)
        
    #     print(f"{class_name:<20} "
    #           f"{class_stats['gt_pixels']:<12} "
    #           f"{class_stats['pred_pixels']:<12} "
    #           f"{ratio:.3f}")

def save_results_to_csv(results, save_path, labels):
    """Save detailed results to CSV files"""
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Per-file results
    file_data = []
    for file_result in results['per_file_metrics']:
        row = {
            'filename': file_result['filename'],
            'mean_dice': file_result['mean_dice'],
            'mean_iou': file_result['mean_iou'],
            'mean_hd': file_result['mean_hd']

        }
        # Add per-class results
        for class_label in labels:
            class_name = CLASS_NAMES[class_label].replace(' ', '_')
            if class_label in file_result['class_results']:
                row[f'{class_name}_dice'] = file_result['class_results'][class_label]['dice']
                row[f'{class_name}_iou'] = file_result['class_results'][class_label]['iou']
                row[f'{class_name}_hd'] = file_result['class_results'][class_label]['hd']
            else:
                row[f'{class_name}_dice'] = 0.0
                row[f'{class_name}_iou'] = 0.0
                row[f'{class_name}_hd'] = 0.0
        file_data.append(row)
    
    df = pd.DataFrame(file_data)
    df.to_csv(save_path, index=False)
    print(f"\nDetailed results saved to: {save_path}")

# Example usage function
def evaluate_segmentation_results(gt_dir, pred_dir, labels=None, save_csv=None):
    """
    Main function to evaluate segmentation results
    
    Parameters:
    -----------
    input_base_dir : str
        Base directory containing GT masks
    output_dir : str  
        Directory containing predicted masks
    labels : list, optional
        Classes to evaluate. Default: [1,2,3,4,5,6,7]
    save_csv : str, optional
        Path to save CSV results
    
    Returns:
    --------
    dict : Comprehensive evaluation results
    """
    
    # gt_dir = Path(mask_base_dir) /
    # pred_dir = Path(pred_base_dir) / "imgs" 
    
    if labels is None:
        labels = [1, 2, 3, 4, 5, 6, 7]
    
    print(f"GT Directory: {gt_dir}")
    print(f"Prediction Directory: {pred_dir}")
    
    results = compute_comprehensive_metrics(
        gt_dir=gt_dir,
        pred_dir=pred_dir, 
        labels=labels,
        verbose=True,
        save_results=save_csv
    )
    
    return results


# mask_path = Path(f"{input_base_dir}")       # GT masks
# img_path = Path(f"{input_base_dir}/imgs_png")     # Original images
# pre_masks_dir = Path(f"{OUTPUT_DIR}/imgs")        # Predicted masks

# USAGE EXAMPLE:

if __name__ == "__main__":
    # Set your directories here
    input_base_dir = mask_path  # Contains masks/ subdirectory
    output_dir = "./"     # Contains imgs/ subdirectory
    
    # Option 1: Evaluate all classes
    results = evaluate_segmentation_results(
        gt_dir=Path(f"{input_base_dir}") ,
        pred_dir=Path(f"{OUTPUT_DIR}/imgs"),
        labels=[1, 2, 3, 4, 5, 6, 7],  # All classes
        save_csv="evaluation_results.csv"
    )
    
    # Option 2: Evaluate specific classes only
    # results = evaluate_segmentation_results(
    #     input_base_dir=input_base_dir,
    #     output_dir=output_dir,
    #     labels=[1, 2, 4],  # Only Femur, Tibia, Femoral Cartilage
    #     save_csv="specific_classes_results.csv"
    # )
    
    # Access specific results
    print(f"Overall Mean Dice: {results['overall_metrics']['mean_dice']:.4f}")
    print(f"Femur Dice: {results['per_class_metrics'][1]['mean_dice']:.4f}")

In [ ]:
input_base_dir